# KoMA-RAG ablation study (Kaggle runner)

Runs `run_ablation.py` on Kaggle. Before running:

1. **Settings (right sidebar) -> Internet: On** (required to reach the GitHub repo and the Mistral API).
2. **Add-ons -> Secrets -> Add a new secret**, name it `MISTRAL_API_KEY`, paste your Mistral key, attach it to this notebook.
3. Push your local fixes to GitHub first -- this notebook clones from `origin`, branch `KoMA-V2`. If you haven't pushed yet, `git clone` below will pull the *old* code without the latest fixes (crash-hardening, Mistral provider switch, retry/instrumentation).
4. No GPU needed -- everything here runs on CPU, so leave the accelerator off and save your GPU-hours quota.

**Session length:** Kaggle notebook sessions have a runtime cap (historically ~9-12h; check your current limit under Settings). At Mistral's ~7s/call, the full batch (10 seed + 20 episodes x 4 configs, full 20-step episodes) is roughly 3.5-7h -- should fit in one session, but if it doesn't, use the `--only=<phase>` flag below to run one phase per session and click **Save Version** between phases -- `run_ablation.py` skips phases that already finished, so re-running the whole notebook after a restart just picks up where it left off. Phase keys: `seed`, `base_koma`, `koma_master`, `koma_verification`, `koma_rag_full`.

**Python version note:** this repo's `requirements.txt` pins old package versions (numpy 1.24.3, gymnasium 0.28.1) that only ship prebuilt wheels up to Python 3.11 -- on Python 3.12+ pip has to build numpy from source and fails without a build backend. Cell 2 below prints the Kaggle image's Python version so you can check this before the install runs; if it's 3.12+ and the install fails with a `setuptools.build_meta` error, that's this exact issue -- let me know and I'll adjust the pins.

In [ ]:
import os

REPO_URL = "https://github.com/hasnain1241/KoMA-RAG.git"
BRANCH = "KoMA-V2"
REPO_DIR = "/kaggle/working/KoMA-RAG"

if not os.path.exists(REPO_DIR):
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull
%cd {REPO_DIR}

In [ ]:
# Kaggle's base image already has numpy/pandas/etc; -q keeps output short.
# Expect some dependency-resolver warnings against Kaggle's preinstalled
# versions -- generally safe to ignore for a short-lived kernel.
!python --version
!pip install -q -r requirements.txt

In [ ]:
# Mistral key from Kaggle Secrets (never paste the key directly into a cell).
from kaggle_secrets import UserSecretsClient
os.environ["MISTRAL_API_KEY"] = UserSecretsClient().get_secret("MISTRAL_API_KEY")

# Kaggle has no display server; highway_env/pygame need a dummy SDL driver
# to render frames off-screen (see highway_env/envs/common/graphics.py).
os.environ["SDL_VIDEODRIVER"] = "dummy"

In [ ]:
# Full unattended run (all phases, aggregates at the end).
# If you're worried about the session time limit, comment this out and
# use the per-phase cell below instead.
!python run_ablation.py

## Alternative: one phase per session

Run this cell instead of the one above, changing `PHASE` each session, then click **Save Version** so `/kaggle/working/KoMA-RAG/result/` is preserved before the session ends.

In [ ]:
PHASE = "seed"  # seed -> base_koma -> koma_master -> koma_verification -> koma_rag_full
!python run_ablation.py --only={PHASE}

In [ ]:
# Once all 4 ablation configs have finished (in any number of sessions),
# rebuild the mean +/- std table without re-running anything:
!python run_ablation.py --aggregate-only
print(open("result/ablation_summary.md").read())